In [1]:
import pandas as pd
import hashlib

In [2]:
df = pd.read_csv('/Users/eamonmcentee/Documents/sales_data_sample.csv', encoding='latin1')

# Note - will set this up at a project level
project = 'CRM'

In [3]:
print(df.head(10))

   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
3        10145               45      83.26                6  3746.70   
4        10159               49     100.00               14  5205.27   
5        10168               36      96.66                1  3479.76   
6        10180               29      86.13                9  2497.77   
7        10188               48     100.00                1  5512.32   
8        10201               22      98.57                2  2168.54   
9        10211               41     100.00               14  4708.44   

         ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   2/24/2003 0:00  Shipped       1         2     2003  ...   
1    5/7/2003 0:00  Shipped       2         5     2003  ...   
2    7/1/2003 0:00

In [4]:
# Edit specific row and column
df.at[0, 'ADDRESSLINE1'] = '1234 New Address'
print(df.at[0, 'ADDRESSLINE1'])

1234 New Address


In [5]:
# set index to a generated unique id 
import uuid
df['UUID'] = [str(uuid.uuid1()) for _ in range(len(df))]
df.set_index(df['UUID'], inplace=True)

In [6]:
print(df.head(10))
# return row with specific index
indx = df.index[0]
row =  df.loc[indx]
print(row['ADDRESSLINE1'])

                                      ORDERNUMBER  QUANTITYORDERED  PRICEEACH  \
UUID                                                                            
06380ab6-2e9f-11f0-be78-6ada60f41393        10107               30      95.70   
06380d04-2e9f-11f0-be78-6ada60f41393        10121               34      81.35   
06380d40-2e9f-11f0-be78-6ada60f41393        10134               41      94.74   
06380d68-2e9f-11f0-be78-6ada60f41393        10145               45      83.26   
06380d90-2e9f-11f0-be78-6ada60f41393        10159               49     100.00   
06380dae-2e9f-11f0-be78-6ada60f41393        10168               36      96.66   
06380dcc-2e9f-11f0-be78-6ada60f41393        10180               29      86.13   
06380dea-2e9f-11f0-be78-6ada60f41393        10188               48     100.00   
06380e08-2e9f-11f0-be78-6ada60f41393        10201               22      98.57   
06380e26-2e9f-11f0-be78-6ada60f41393        10211               41     100.00   

                           

In [7]:
# Next steps : start creating functions to perform actions on the data - edit/change columns and record history of changes
print(uuid.uuid1())

0bc94864-2e9f-11f0-be78-6ada60f41393


In [8]:
# Retrieve Row -
def retrieve_row (df, index):
        row = df.loc[index]
        return row
    

In [9]:
# Edit Row values - update row with new value and return old row for archive
def edit_row_column (row, column, new_value):
        old_row = row.copy()
        row[column] = new_value
        return row, old_row
    

In [10]:
row = retrieve_row(df, indx)

In [11]:
print(row)

ORDERNUMBER                                        10107
QUANTITYORDERED                                       30
PRICEEACH                                           95.7
ORDERLINENUMBER                                        2
SALES                                             2871.0
ORDERDATE                                 2/24/2003 0:00
STATUS                                           Shipped
QTR_ID                                                 1
MONTH_ID                                               2
YEAR_ID                                             2003
PRODUCTLINE                                  Motorcycles
MSRP                                                  95
PRODUCTCODE                                     S10_1678
CUSTOMERNAME                           Land of Toys Inc.
PHONE                                         2125557818
ADDRESSLINE1                            1234 New Address
ADDRESSLINE2                                         NaN
CITY                           

In [12]:
x, y = edit_row_column(row, 'ADDRESSLINE1', '13 Ballinteer')

/var/folders/qw/p4ztp46x00x8z6wtr3d6sjg80000gn/T/ipykernel_49000/3925772620.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row[column] = new_value


In [13]:
print(y)

ORDERNUMBER                                        10107
QUANTITYORDERED                                       30
PRICEEACH                                           95.7
ORDERLINENUMBER                                        2
SALES                                             2871.0
ORDERDATE                                 2/24/2003 0:00
STATUS                                           Shipped
QTR_ID                                                 1
MONTH_ID                                               2
YEAR_ID                                             2003
PRODUCTLINE                                  Motorcycles
MSRP                                                  95
PRODUCTCODE                                     S10_1678
CUSTOMERNAME                           Land of Toys Inc.
PHONE                                         2125557818
ADDRESSLINE1                            1234 New Address
ADDRESSLINE2                                         NaN
CITY                           

In [14]:
# Hash the values of a datafame row
def hash_row(row):
    # Convert the row to a string representation
    row_str = row.to_json()
    # Create a SHA-256 hash of the string
    return hashlib.sha256(row_str.encode()).hexdigest()

In [15]:
sha_value = hash_row(row)

In [16]:
sha_value[0:2]
sha_value[2:]

'3399e7e7d257689737ca859a671edc0158464095e4a3c9ee96648bcc93d315'

In [ ]:
import os
import json
cwd = os.getcwd()



def save_row_to_directory(row, sha_value, project):
    directory_path = './objects' + '/' + project 
    # Create the directory path using the project name
    directory_name = sha_value[0:2]
    file_name = sha_value[2:] + '.json'


    # Check if the object directory exists
    if not os.path.exists(directory_path):
        # Create the directory
        os.makedirs(directory_path)
        print(f"Directory created: {directory_path}")
    else:
        print(f"Directory already exists: {directory_path}")

    # Check if the subdirectory exists
    full_directory_path = os.path.join(directory_path, directory_name)

    if not os.path.exists(full_directory_path):
        # Create the directory
        os.makedirs(full_directory_path)
        print(f"Directory created: {full_directory_path}")
    else:
        print(f"Directory already exists: {full_directory_path}")
    
    # Convert the row to JSON
    row_json = row.to_json()
    
    # Define the full file path
    file_path = os.path.join(full_directory_path, file_name)
    
    # Write the JSON to the file
    with open(file_path, 'w') as file:
        file.write(row_json)
    
    print(f"Row saved to {file_path}")

# Example usage

save_row_to_directory(row, sha_value, project)

Directory already exists: ./objects/CRM
Directory created: ./objects/CRM/9d
Row saved to ./objects/CRM/9d/3399e7e7d257689737ca859a671edc0158464095e4a3c9ee96648bcc93d315.json


In [18]:
cwd = os.getcwd()
print(cwd)

/Users/eamonmcentee/VSCode
